<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Restrictions" data-toc-modified-id="Restrictions-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Restrictions</a></span><ul class="toc-item"><li><span><a href="#limit" data-toc-modified-id="limit-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>limit</a></span></li><li><span><a href="#distinct" data-toc-modified-id="distinct-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>distinct</a></span><ul class="toc-item"><li><span><a href="#where" data-toc-modified-id="where-1.2.1"><span class="toc-item-num">1.2.1&nbsp;&nbsp;</span>where</a></span></li></ul></li></ul></li><li><span><a href="#Colonnes" data-toc-modified-id="Colonnes-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Colonnes</a></span></li><li><span><a href="#expr" data-toc-modified-id="expr-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>expr</a></span></li><li><span><a href="#Transformations" data-toc-modified-id="Transformations-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Transformations</a></span><ul class="toc-item"><li><span><a href="#Projections" data-toc-modified-id="Projections-4.1"><span class="toc-item-num">4.1&nbsp;&nbsp;</span>Projections</a></span><ul class="toc-item"><li><span><a href="#selectExpr" data-toc-modified-id="selectExpr-4.1.1"><span class="toc-item-num">4.1.1&nbsp;&nbsp;</span>selectExpr</a></span></li><li><span><a href="#drop" data-toc-modified-id="drop-4.1.2"><span class="toc-item-num">4.1.2&nbsp;&nbsp;</span>drop</a></span></li><li><span><a href="#orderBy" data-toc-modified-id="orderBy-4.1.3"><span class="toc-item-num">4.1.3&nbsp;&nbsp;</span>orderBy</a></span></li></ul></li></ul></li></ul></div>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("02-API.DataFrames-Restrictions")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/09/24 09:51:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 09:51:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/24 09:51:53 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20260924095152-0014/events_1_app-20260924095152-0014.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20260924095152-0014
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

In [4]:
print(f"spark.executor.cores = {spark.conf.get('spark.executor.cores')}\nspark.executor.memory = {spark.conf.get('spark.executor.memory')}")

spark.executor.cores = 1
spark.executor.memory = 1g


# Restrictions
Une Transformation qui permet de limiter le nombre d’enregistrements du DataFrame résultante est une restriction.

In [5]:
from pyspark.sql.functions import *

meteoDataFrame  = spark.read.format('csv')\
    .option('sep',';')\
    .option('header','true')\
    .option('nullValue','mq')\
    .option('inferSchema', 'true')\
    .load('../data/meteo/')  #   .cache()


from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType

schema = StructType([
        StructField('Id'           , StringType() , True),
        StructField('ville'        , StringType() , True),
        StructField('latitude'     , FloatType() , True),
        StructField('longitude'    , FloatType() , True),
        StructField('altitude'     , IntegerType() , True)])

villes  = spark.read.format('csv')   \
      .option('sep',';')                \
      .option('mergeSchema', 'true')    \
      .option('header','true')          \
      .schema(schema)                   \
      .load('../data/postesSynop.csv') #      .cache()

meteoDataFrame.count()

525327

## limit

In [6]:
meteoDataFrame.selectExpr(
        'numer_sta','t  - 273.15 as temperature'
        ).limit(5).show()

+---------+------------------+
|numer_sta|       temperature|
+---------+------------------+
|     7005|13.100000000000023|
|     7015|              14.5|
|     7020| 8.800000000000011|
|     7027| 9.200000000000045|
|     7037|12.600000000000023|
+---------+------------------+



## distinct

In [7]:
meteoDataFrame.select('numer_sta').distinct().count()

62

In [8]:
meteoDataFrame.select('numer_sta').distinct().show(3)

[Stage 13:=============================>                            (2 + 2) / 4]

+---------+
|numer_sta|
+---------+
|     7005|
|     7020|
|     7027|
+---------+
only showing top 3 rows


In [9]:
meteoDataFrame.distinct().count()

26/09/24 09:52:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

525327

### where

In [10]:
villes.where('latitude > 49').show()

+-----+---------------+---------+---------+--------+
|   Id|          ville| latitude|longitude|altitude|
+-----+---------------+---------+---------+--------+
|07005|      ABBEVILLE|   50.136|    1.834|      69|
|07015|  LILLE-LESQUIN|    50.57|   3.0975|      47|
|07020|PTE DE LA HAGUE|49.725166|-1.939833|       6|
|07027| CAEN-CARPIQUET|    49.18|-0.456167|      67|
|07037|     ROUEN-BOOS|   49.383| 1.181667|     151|
|07072|   REIMS-PRUNAY|49.209667| 4.155333|      95|
+-----+---------------+---------+---------+--------+



In [11]:
villes.where('latitude > 49 and altitude > 90').show()

+-----+------------+---------+---------+--------+
|   Id|       ville| latitude|longitude|altitude|
+-----+------------+---------+---------+--------+
|07037|  ROUEN-BOOS|   49.383| 1.181667|     151|
|07072|REIMS-PRUNAY|49.209667| 4.155333|      95|
+-----+------------+---------+---------+--------+



In [12]:
villes.where('latitude > 49 or altitude > 800').show()

+-----+---------------+---------+---------+--------+
|   Id|          ville| latitude|longitude|altitude|
+-----+---------------+---------+---------+--------+
|07005|      ABBEVILLE|   50.136|    1.834|      69|
|07015|  LILLE-LESQUIN|    50.57|   3.0975|      47|
|07020|PTE DE LA HAGUE|49.725166|-1.939833|       6|
|07027| CAEN-CARPIQUET|    49.18|-0.456167|      67|
|07037|     ROUEN-BOOS|   49.383| 1.181667|     151|
|07072|   REIMS-PRUNAY|49.209667| 4.155333|      95|
|07471|  LE PUY-LOUDES|  45.0745|    3.764|     833|
|07591|         EMBRUN|44.565666| 6.502333|     871|
+-----+---------------+---------+---------+--------+



In [13]:
meteoDataFrame.count(), meteoDataFrame.sample(1/100).count(), meteoDataFrame.sample(True,1/100).count()

(525327, 5149, 5293)

In [14]:
villes.sample(True,2/100,0).show()

+-----+---------------+---------+---------+--------+
|   Id|          ville| latitude|longitude|altitude|
+-----+---------------+---------+---------+--------+
|07168|TROYES-BARBEREY| 48.32467|     4.02|     112|
|07280|  DIJON-LONGVIC|47.267834| 5.088333|     219|
+-----+---------------+---------+---------+--------+



In [15]:
villes.filter("Id == '07168' or Id == '07280'").show()

+-----+---------------+---------+---------+--------+
|   Id|          ville| latitude|longitude|altitude|
+-----+---------------+---------+---------+--------+
|07168|TROYES-BARBEREY| 48.32467|     4.02|     112|
|07280|  DIJON-LONGVIC|47.267834| 5.088333|     219|
+-----+---------------+---------+---------+--------+



In [16]:
villes.filter("Id = '07168' or Id = '07280'").show()

+-----+---------------+---------+---------+--------+
|   Id|          ville| latitude|longitude|altitude|
+-----+---------------+---------+---------+--------+
|07168|TROYES-BARBEREY| 48.32467|     4.02|     112|
|07280|  DIJON-LONGVIC|47.267834| 5.088333|     219|
+-----+---------------+---------+---------+--------+

